# L07-01｜LoRA 实验导学


## 本 Notebook 做什么

本 Notebook 负责准备 LoRA 实验的起点：看清哪些参数会更新，在 ModelArts 的 Ascend 环境里准备 `ms-swift`、模型和训练数据，并记录本次运行会用到的路径。

这里只做预检，不评价模型效果。单独一条 loss 曲线或资源记录，不能说明模型效果、吞吐量或昇腾性能。


## 环境准备

下面的代码单元不依赖仓库里的其他文件。它检查 Ascend NPU，按当前 Python 解释器补装 `modelscope`、`ms-swift` 和绘图库，下载 `Qwen/Qwen3-0.6B`，再生成一份小型 JSONL 训练集。“纯净环境”指 Python 3.10–3.12 且带 Ascend/CANN/`torch_npu` 运行时的 ModelArts 镜像；驱动不是可以用 pip 临时补上的 Python 包。每个 Notebook 都有自己的初始化代码，可以单独复制到新的 ModelArts 实例运行。


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import shlex
import shutil
import site
import subprocess
import sys
from pathlib import Path

def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)

require((3, 10) <= sys.version_info[:2] <= (3, 12), '本流程需要 Python 3.10、3.11 或 3.12；请使用带匹配 Ascend 运行时的 ModelArts 镜像。')
packages = {'modelscope': 'modelscope', 'swift': 'ms-swift', 'matplotlib': 'matplotlib'}
missing = [dist for module, dist in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print('安装缺失依赖：', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', *missing])
python_bin = str(Path(sys.executable).parent)
user_bin = str(Path(site.getuserbase()) / 'bin')
old_path = os.environ.get('PATH', '')
path_entries = old_path.split(os.pathsep) if old_path else []
for candidate in (python_bin, user_bin):
    if candidate not in path_entries:
        old_path = candidate + os.pathsep + old_path
        path_entries.insert(0, candidate)
os.environ['PATH'] = old_path

def load_ascend_env() -> None:
    candidates = [
        Path('/usr/local/Ascend/ascend-toolkit/set_env.sh'),
        Path('/usr/local/Ascend/ascend-toolkit/latest/set_env.sh'),
    ]
    ascend_root = Path('/usr/local/Ascend')
    if ascend_root.is_dir():
        candidates.extend(sorted(ascend_root.glob('**/set_env.sh')))
    for script in candidates:
        if not script.is_file():
            continue
        result = subprocess.run(
            ['bash', '-lc', f'source {shlex.quote(str(script))} >/dev/null 2>&1 && env -0'],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False,
        )
        if result.returncode != 0:
            continue
        for item in result.stdout.split(b'\0'):
            if b'=' in item:
                key, value = item.split(b'=', 1)
                os.environ[key.decode()] = value.decode(errors='ignore')
        print('已加载 Ascend 环境：', script)
        return
    print('未找到 CANN set_env.sh；继续使用当前 ModelArts 进程环境。')

load_ascend_env()
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')
import torch
try:
    import torch_npu  # noqa: F401
except Exception as exc:
    raise RuntimeError('当前环境无法导入 torch_npu。请使用带 Ascend/CANN 运行时的 ModelArts 镜像。') from exc
require(hasattr(torch, 'npu'), '当前 PyTorch 没有 torch.npu；请检查 ModelArts 的 Ascend 运行时。')
npu_count = torch.npu.device_count()
require(npu_count > 0, '没有检测到 NPU。请确认 ModelArts 实例规格和可见设备。')
torch_version = getattr(torch, '__version__', 'unknown')
torch_npu_version = getattr(torch_npu, '__version__', 'unknown')
def major_minor(version: str) -> tuple[int, int] | None:
    try:
        parts = version.split('+', 1)[0].split('.')
        return int(parts[0]), int(parts[1])
    except (IndexError, ValueError):
        return None
if major_minor(torch_version) and major_minor(torch_npu_version):
    require(major_minor(torch_version) == major_minor(torch_npu_version), f'torch 与 torch_npu 版本不匹配：{torch_version} vs {torch_npu_version}。请按同一套 CANN/PyTorch/torch_npu 重新准备 ModelArts 镜像。')
torch.npu.set_device(0)
_ = torch.zeros(1, device='npu:0')

NOTEBOOK_ID = 'L07-01'
WORK_DIR = Path(os.environ.get('L07_WORK_DIR', str(Path.cwd() / f'{NOTEBOOK_ID}_workspace'))).expanduser()
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ID = os.environ.get('L07_MODEL_ID', 'Qwen/Qwen3-0.6B')
MODEL_CACHE = Path(os.environ.get('MODELSCOPE_CACHE', str(WORK_DIR / 'modelscope_cache'))).expanduser()
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
from modelscope import snapshot_download
MODEL_PATH = Path(snapshot_download(MODEL_ID, cache_dir=str(MODEL_CACHE)))
require((MODEL_PATH / 'config.json').is_file(), f'模型目录缺少 config.json：{MODEL_PATH}')

def qwen3_answer(text: str) -> str:
    return '<think>\n\n</think>\n\n' + text

examples = [
    ('请用一句话解释 LoRA 与全参数微调的区别。', 'LoRA 只训练注入的低秩适配器参数，全参数微调会更新模型的全部参数。'),
    ('Python 中如何获取列表长度？', '使用内置函数 len，例如 len([1, 2, 3]) 的结果是 3。'),
    ('一个 batch 有 2 条样本，梯度累积 4 步，完成一次更新前处理多少条样本？', '在没有丢弃样本的情况下，会先处理 2 乘以 4，也就是 8 条样本。'),
    ('把“先检查日志，再判断原因”翻译成英文。', 'Check the logs first, then identify the cause.'),
]
records = [
    {'messages': [
        {'role': 'system', 'content': '你是一个简洁、准确的课程实验助手。'},
        {'role': 'user', 'content': question + ' /no_think'},
        {'role': 'assistant', 'content': qwen3_answer(answer)},
    ]}
    for question, answer in examples
]
TRAIN_DATA = WORK_DIR / 'train.jsonl'
with TRAIN_DATA.open('w', encoding='utf-8') as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
loaded_records = [json.loads(line) for line in TRAIN_DATA.read_text(encoding='utf-8').splitlines() if line.strip()]
require(len(loaded_records) == len(records), '生成的 JSONL 条数不一致。')
require(all('messages' in item for item in loaded_records), '每条训练数据都必须包含 messages。')
require(shutil.which('swift') is not None, '找不到 swift 命令；请确认 ms-swift 已安装且当前 Python 的 bin 目录在 PATH 中。')
environment_record = {'python': sys.version.split()[0], 'torch': getattr(torch, '__version__', 'unknown'), 'torch_npu': getattr(torch_npu, '__version__', 'unknown'), 'npu_count': npu_count, 'visible_npus': os.environ['ASCEND_RT_VISIBLE_DEVICES'], 'model_id': MODEL_ID, 'model_path': str(MODEL_PATH), 'train_data': str(TRAIN_DATA)}
(WORK_DIR / 'environment.json').write_text(json.dumps(environment_record, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print({'model_id': MODEL_ID, 'model_path': str(MODEL_PATH), 'train_data': str(TRAIN_DATA), 'npu_count': npu_count})
print('导学预检完成：依赖、NPU、模型和本地 JSONL 数据均已准备。')


## 读懂几个参数

### LoRA 配置

- `lora_rank` 是低秩分支的维度。值越大，适配器通常越大。
- `lora_alpha` 是这条分支的缩放系数，不是学习率。
- `target_modules` 指定哪些线性层接收 LoRA 适配器。实际注入了哪些层，要以训练输出为准。
- 学习率、batch size、最大序列长度和梯度累积一起影响一次运行。只改其中一个参数时，才比较容易看出变化来自哪里。

### 实验记录

每次训练至少保存模型和数据路径、配置、启动命令、环境信息、`logging.jsonl`、checkpoint / adapter 路径和异常记录。图表只是日志的可视化，不能代替原始日志。


## 读图时分开三种说法

| 层次 | 例子 | 推荐表达 |
| --- | --- | --- |
| 直接观察 | 第 100 step 的日志记录了一个 loss 值 | “日志显示……” |
| 合理解释 | loss 波动可能与学习率、batch 或数据顺序有关 | “可能与……有关，需要进一步核验……” |
| 不可直接推出 | “该配置一定更好”“模型已经泛化” | 不能仅凭单次训练 loss 得出 |

L07-03 会把这三层分别写出来：日志事实、可能解释，以及暂时不能证明的结论。


## 出问题时先查哪里

| 现象 | 首先检查 | 不建议直接做什么 |
| --- | --- | --- |
| 找不到 NPU | ModelArts 规格、kernel 和 `torch_npu` | 直接安装一个不匹配的 torch_npu |
| 找不到模型或数据 | 当前实例的实际路径与权限 | 照抄别人的绝对路径 |
| 训练一启动就退出 | `notebook_stdout.log` 的最后一段 | 一次改很多超参数 |
| 没有 loss 日志 | 输出目录和 `logging.jsonl` | 手工补一条“应该有的曲线” |
| loss 有波动 | step、学习率、日志间隔和异常 | 只看一个点就下结论 |


## 运行前想清楚

1. smoke test 只跑几步，为什么不能代替完整训练？
2. loss 下降了，为什么还不能说模型效果更好？
3. 两次训练的 batch size 或最大序列长度不同，为什么不能直接比较显存峰值？

想清楚这三点，再进入 L07-02。


## 开始 L07-02 前

进入 L07-02 前，用自己的话回答下面的问题：

- LoRA 与全参数微调在“哪些参数会更新”上有什么不同？
- smoke test 的成功标志是什么？为什么它不是“loss 足够低”？
- 你准备把本次训练的日志和输出保存在哪里？

如果有一题答不上来，回看上面的参数和证据说明，再继续运行。
